# Week 9 Submission: Improving GRPO with Better Reward Design

This notebook extends the Week 9 GRPO lab by:
1. Analysing the existing reward baselines and explaining their limitations
2. Designing a new multi-signal reward function
3. Running the GRPO training pipeline with the improved reward
4. Evaluating and comparing results against the provided baselines

## Installation

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    pass
#@title Colab Extra Install { display-mode: "form" }
import os
import json
import re
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
    !uv pip install -qqq --upgrade \
        unsloth {get_vllm} {get_numpy} {get_pil} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

## Imports and Model Configuration

In [2]:
import os
import json
import re
import gc
import shutil
import difflib

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import TextStreamer, AutoTokenizer
from unsloth import FastLanguageModel

os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

BASE_MODEL_NAME = "unsloth/Qwen3-4B-Base"
max_seq_length = 2048
lora_rank = 32

/tmp/ipykernel_3732/927413960.py:13: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 04-03 08:33:34 [__init__.py:244] Automatically detected platform cuda.
ERROR 04-03 08:33:39 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!


## Google Drive Setup

In [4]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/grpo_lab_checkpoints"
os.makedirs(DRIVE_ROOT, exist_ok=True)

Mounted at /content/drive


## Checkpoint Helpers

In [5]:
def get_latest_checkpoint(checkpoint_root: str):
    if not os.path.isdir(checkpoint_root):
        return None
    checkpoint_dirs = []
    for name in os.listdir(checkpoint_root):
        full_path = os.path.join(checkpoint_root, name)
        if os.path.isdir(full_path) and re.fullmatch(r"checkpoint-\d+", name):
            step = int(name.split("-")[-1])
            checkpoint_dirs.append((step, full_path))
    if not checkpoint_dirs:
        return None
    checkpoint_dirs.sort(key=lambda x: x[0])
    return checkpoint_dirs[-1][1]


def get_checkpoint_step(checkpoint_dir: str) -> int:
    if checkpoint_dir is None:
        return 0
    trainer_state_path = os.path.join(checkpoint_dir, "trainer_state.json")
    if os.path.exists(trainer_state_path):
        with open(trainer_state_path, "r") as f:
            state = json.load(f)
        return int(state.get("global_step", 0))
    match = re.search(r"checkpoint-(\d+)$", checkpoint_dir)
    if match is not None:
        return int(match.group(1))
    return 0

## Copy Shared Checkpoints (if available)

In [6]:
SHARED_DIR = "/content/drive/MyDrive/Shared checkpoints/grpo_lab_checkpoints"

EXPECTED_FOLDERS = [
    "sft_only_baseline",
    "grpo_better_binary_reward",
    "grpo_bad_has_number",
    "grpo_bad_format_only"
]

if os.path.exists(SHARED_DIR) and os.path.isdir(SHARED_DIR):
    print(f"Found shared folder: {SHARED_DIR}")
    available_folders = []
    missing_folders = []
    for folder_name in EXPECTED_FOLDERS:
        src = os.path.join(SHARED_DIR, folder_name)
        if os.path.exists(src) and os.path.isdir(src):
            available_folders.append(folder_name)
        else:
            missing_folders.append(folder_name)
    if available_folders:
        print("Available checkpoint folders in shared directory:")
        for folder_name in available_folders:
            print(f"  - {folder_name}")
        if missing_folders:
            print("Missing expected folders:")
            for folder_name in missing_folders:
                print(f"  - {folder_name}")
        for folder_name in available_folders:
            src = os.path.join(SHARED_DIR, folder_name)
            dst = os.path.join(DRIVE_ROOT, folder_name)
            if os.path.exists(dst):
                print(f"[SKIP] Already exists in your Drive: {dst}")
            else:
                print(f"[COPY] {src} -> {dst}")
                shutil.copytree(src, dst)
        print("Checkpoint copy step complete.")
    else:
        print("Shared folder exists, but no expected checkpoint folders were found.")
        print("Proceeding with an empty checkpoint directory for now.")
else:
    print(f"Shared folder not found: {SHARED_DIR}")
    print("Proceeding with an empty checkpoint directory for now.")

print(f"Using DRIVE_ROOT: {DRIVE_ROOT}")

Found shared folder: /content/drive/MyDrive/Shared checkpoints/grpo_lab_checkpoints
Available checkpoint folders in shared directory:
  - sft_only_baseline
  - grpo_better_binary_reward
  - grpo_bad_has_number
  - grpo_bad_format_only
[SKIP] Already exists in your Drive: /content/drive/MyDrive/grpo_lab_checkpoints/sft_only_baseline
[SKIP] Already exists in your Drive: /content/drive/MyDrive/grpo_lab_checkpoints/grpo_better_binary_reward
[SKIP] Already exists in your Drive: /content/drive/MyDrive/grpo_lab_checkpoints/grpo_bad_has_number
[SKIP] Already exists in your Drive: /content/drive/MyDrive/grpo_lab_checkpoints/grpo_bad_format_only
Checkpoint copy step complete.
Using DRIVE_ROOT: /content/drive/MyDrive/grpo_lab_checkpoints


In [7]:
print("\nCurrent contents of DRIVE_ROOT:")
if os.path.exists(DRIVE_ROOT):
    for name in sorted(os.listdir(DRIVE_ROOT)):
        print("-", name)
else:
    print("DRIVE_ROOT does not exist yet.")


Current contents of DRIVE_ROOT:
- grpo_bad_format_only
- grpo_bad_has_number
- grpo_better_binary_reward
- grpo_better_quality_reward
- sft_only_baseline


In [8]:
SFT_BASELINE_DIR = os.path.join(DRIVE_ROOT, "sft_only_baseline")
os.makedirs(SFT_BASELINE_DIR, exist_ok=True)
print(f"SFT_BASELINE_DIR: {SFT_BASELINE_DIR}")

SFT_BASELINE_DIR: /content/drive/MyDrive/grpo_lab_checkpoints/sft_only_baseline


## Chat Template Setup

In [9]:
if os.path.exists(os.path.join(SFT_BASELINE_DIR, "tokenizer_config.json")):
    TOKENIZER_SOURCE = SFT_BASELINE_DIR
else:
    TOKENIZER_SOURCE = BASE_MODEL_NAME

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_SOURCE)
print(f"Loaded tokenizer from: {TOKENIZER_SOURCE}")

Loaded tokenizer from: /content/drive/MyDrive/grpo_lab_checkpoints/sft_only_baseline


In [10]:
reasoning_start = "<start_working_out>"
reasoning_end   = "<end_working_out>"
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION>'

In [11]:
chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

## SFT Pre-Fine-Tuning (from checkpoint)

In [12]:
dataset = load_dataset("unsloth/OpenMathReasoning-mini", split="cot")
dataset = dataset.to_pandas()[
    ["expected_answer", "problem", "generated_solution"]
]

is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors="coerce").notnull()
dataset = dataset.iloc[np.where(is_number)[0]]
dataset.shape

README.md:   0%|          | 0.00/603 [00:00<?, ?B/s]

data/cot-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

Generating cot split:   0%|          | 0/19252 [00:00<?, ? examples/s]

(7507, 3)

In [13]:
def format_dataset(x):
    expected_answer = x["expected_answer"]
    problem = x["problem"]
    thoughts = x["generated_solution"]
    thoughts = thoughts.replace("<think>", "").replace("</think>", "")
    thoughts = thoughts.strip()
    final_prompt = \
        reasoning_start + thoughts + reasoning_end + \
        solution_start + expected_answer + solution_end
    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : final_prompt},
    ]

dataset["Messages"] = dataset.apply(format_dataset, axis=1)

In [14]:
dataset["N"] = dataset["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))
dataset = dataset.loc[dataset["N"] <= max_seq_length/2].copy()
dataset.shape

(59, 5)

In [15]:
dataset["text"] = tokenizer.apply_chat_template(dataset["Messages"].values.tolist(), tokenize=False)
dataset = Dataset.from_pandas(dataset)
dataset

Dataset({
    features: ['expected_answer', 'problem', 'generated_solution', 'Messages', 'N', 'text', '__index_level_0__'],
    num_rows: 59
})

In [16]:
RESUME_SFT_IF_AVAILABLE = True
SFT_LAB_EXTRA_STEPS = 10
SFT_FRESH_MAX_STEPS = 100
SFT_SAVE_STEPS = 5

In [17]:
SFT_LATEST_CKPT = get_latest_checkpoint(SFT_BASELINE_DIR) if RESUME_SFT_IF_AVAILABLE else None
SFT_START_STEP = get_checkpoint_step(SFT_LATEST_CKPT)
HAS_SFT_BASELINE = os.path.exists(os.path.join(SFT_BASELINE_DIR, "adapter_config.json"))

if HAS_SFT_BASELINE:
    SFT_STAGE_MODE = "skip_sft_use_existing_baseline"
    SFT_TARGET_MAX_STEPS = 0
    SFT_EFFECTIVE_SAVE_STEPS = 0
elif SFT_LATEST_CKPT is not None:
    SFT_TARGET_MAX_STEPS = SFT_START_STEP + SFT_LAB_EXTRA_STEPS
    SFT_EFFECTIVE_SAVE_STEPS = min(SFT_SAVE_STEPS, SFT_LAB_EXTRA_STEPS)
    SFT_STAGE_MODE = "resume_sft_from_checkpoint"
else:
    SFT_TARGET_MAX_STEPS = SFT_FRESH_MAX_STEPS
    SFT_EFFECTIVE_SAVE_STEPS = SFT_SAVE_STEPS
    SFT_STAGE_MODE = "fresh_sft_from_pretrained"

print("=" * 80)
print(f"SFT run mode: {SFT_STAGE_MODE}")
print(f"SFT latest checkpoint: {SFT_LATEST_CKPT}")
print(f"SFT start step: {SFT_START_STEP}")
print(f"SFT target max steps: {SFT_TARGET_MAX_STEPS}")
print(f"SFT save every: {SFT_EFFECTIVE_SAVE_STEPS} steps")
print("=" * 80)

SFT run mode: skip_sft_use_existing_baseline
SFT latest checkpoint: /content/drive/MyDrive/grpo_lab_checkpoints/sft_only_baseline/checkpoint-110
SFT start step: 110
SFT target max steps: 0
SFT save every: 0 steps


In [18]:
from trl import SFTTrainer, SFTConfig

if SFT_STAGE_MODE != "skip_sft_use_existing_baseline":
    sft_args = SFTConfig(
        output_dir = SFT_BASELINE_DIR,
        max_steps = SFT_TARGET_MAX_STEPS,
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1,
        warmup_steps = 5,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        save_steps=SFT_EFFECTIVE_SAVE_STEPS,
        save_total_limit=2,
    )

In [19]:
if SFT_STAGE_MODE == "skip_sft_use_existing_baseline":
    print("Skipping SFT stage.")
    print(f"Using existing SFT baseline at: {SFT_BASELINE_DIR}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = SFT_BASELINE_DIR,
        max_seq_length = max_seq_length,
        load_in_4bit = False,
        fast_inference = False,
        gpu_memory_utilization = 0.9,
    )

elif SFT_STAGE_MODE == "resume_sft_from_checkpoint":
    print("Resuming SFT from the latest checkpoint.")
    print(f"Checkpoint: {SFT_LATEST_CKPT}")
    print(f"Baseline directory: {SFT_BASELINE_DIR}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = SFT_BASELINE_DIR,
        max_seq_length = max_seq_length,
        load_in_4bit = False,
        fast_inference = False,
        gpu_memory_utilization = 0.9,
    )
    sft_trainer = SFTTrainer(
        model = model,
        processing_class = tokenizer,
        train_dataset = dataset,
        args = sft_args,
    )
    sft_trainer.train(resume_from_checkpoint = SFT_LATEST_CKPT)
    model.save_pretrained(SFT_BASELINE_DIR)
    tokenizer.save_pretrained(SFT_BASELINE_DIR)

else:
    print("No SFT baseline or checkpoint found. Starting fresh SFT.")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen3-4B-Base",
        max_seq_length = max_seq_length,
        load_in_4bit = False,
        fast_inference = True,
        max_lora_rank = lora_rank,
        gpu_memory_utilization = 0.9,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r = lora_rank,
        target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha = lora_rank*2,
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )
    sft_trainer = SFTTrainer(
        model = model,
        processing_class = tokenizer,
        train_dataset = dataset,
        args = sft_args,
    )
    sft_trainer.train()
    model.save_pretrained(SFT_BASELINE_DIR)
    tokenizer.save_pretrained(SFT_BASELINE_DIR)

SFT_BASELINE_DESCRIPTION = (
    "Baseline checkpoint after supervised fine-tuning only, before any RL fine-tuning. "
    "Used to isolate the effect of reward-based GRPO training."
)
baseline_metadata = {
    "experiment_name": "sft_only_baseline",
    "label": "SFT-only baseline",
    "description": SFT_BASELINE_DESCRIPTION,
    "reward_functions": [],
}
with open(os.path.join(SFT_BASELINE_DIR, "experiment_metadata.json"), "w") as f:
    json.dump(baseline_metadata, f, indent=2)
print(f"Baseline metadata saved to: {os.path.join(SFT_BASELINE_DIR, 'experiment_metadata.json')}")

# Re-apply the custom chat template since FastLanguageModel.from_pretrained
# returns a fresh tokenizer that may not have it.
tokenizer.chat_template = chat_template

Skipping SFT stage.
Using existing SFT baseline at: /content/drive/MyDrive/grpo_lab_checkpoints/sft_only_baseline
==((====))==  Unsloth 2026.4.1: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

Unsloth 2026.4.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Baseline metadata saved to: /content/drive/MyDrive/grpo_lab_checkpoints/sft_only_baseline/experiment_metadata.json


In [20]:
del dataset
gc.collect()
torch.cuda.empty_cache()

## GRPO Dataset Preparation

In [21]:
dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split="train")
dataset

README.md: 0.00B [00:00, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/5.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14116 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'solution', 'data_source', 'source_prompt', 'ability', 'reward_model', 'extra_info'],
    num_rows: 14116
})

In [22]:
def extract_hash_answer(text):
    return text

dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["prompt"]},
    ],
    "answer": extract_hash_answer(x["solution"]),
})
dataset[0]

Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

{'prompt': [{'content': 'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION>',
   'role': 'system'},
  {'content': 'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.',
   'role': 'user'}],
 'solution': '34',
 'data_source': 'math_dapo',
 'source_prompt': [{'content': 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\nIn triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $

In [23]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt=True, tokenize=True)},
    batched=True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

You are given a problem.
Think about the problem and provide your working out.
Place it between <start_working_out> and <end_working_out>.
Then, provide your solution between <SOLUTION></SOLUTION><|endoftext|>In triangle $ABC$, $\sin \angle A = \frac{4}{5}$ and $\angle A < 90^\circ$. Let $D$ be a point outside triangle $ABC$ such that $\angle BAD = \angle DAC$ and $\angle BDC = 90^\circ$. Suppose that $AD = 1$ and that $\frac{BD}{CD} = \frac{3}{2}$. If $AB + AC$ can be expressed in the form $\frac{a\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.<start_working_out>


Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

Max Length =  201


## Format Regex

In [24]:
solution_end_regex = r"</SOLUTION>[\s]{0,}" + \
    "(?:" + re.escape(tokenizer.eos_token) + ")?"

match_format = re.compile(
    rf"{reasoning_end}.*?"\
    rf"{solution_start}(.+?){solution_end_regex}"\
    rf"[\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)
match_format

re.compile(r'<end_working_out>.*?<SOLUTION>(.+?)</SOLUTION>[\s]{0,}(?:<\|endoftext\|>)?[\s]{0,}$',
re.MULTILINE|re.DOTALL|re.UNICODE)

In [25]:
match_format.findall(
    "Let me think!<end_working_out>"\
    f"<SOLUTION>\n2\n</SOLUTION>",
)

['\n2\n']

---

## Submission Task 1: Analyse the existing reward baselines and explain their limitations

The notebook provides three reward baselines. Below is each one, with its limitations:

### Bad Reward 1: Format-Only Reward (`match_format_exactly`)

This reward gives +3.0 if the response matches the expected `<end_working_out>...<SOLUTION>...</SOLUTION>` format, and 0 otherwise.

**Limitations:**
- **No correctness signal.** The model receives full reward for a beautifully structured response that contains a completely wrong answer. This decouples the reward from the actual task objective (solving math problems).
- **Easy to game.** The model can learn to always emit the format tokens with an arbitrary number inside `<SOLUTION>` tags and receive maximum reward every time. After a few GRPO steps the model will converge on always producing the template, regardless of reasoning quality.
- **Binary and uninformative.** The reward is all-or-nothing (0 or 3). There is no gradient between a partially correct response and a completely wrong one, so GRPO gets very little useful signal for distinguishing better from worse completions within a generation batch.

### Bad Reward 2: Has-a-Number Reward (`reward_has_number`)

This reward gives +1.0 if the response contains any digit, and 0 otherwise.

**Limitations:**
- **Extremely weak alignment.** Almost any mathematical response will contain at least one digit. In practice, this reward is nearly always 1.0 for all completions, making it useless as a differentiating signal for GRPO (the advantage estimate becomes zero when all completions in a group receive the same reward).
- **Trivially gameable.** The model just needs to output a single digit anywhere. It could output "42" with no reasoning and score the same as a perfect solution.
- **No format or correctness component.** It neither checks structure nor answer accuracy.

### Slightly Better: Binary Correctness Reward (`check_answer_binary`)

This reward gives 1.0 for an exact string match between the extracted answer and the ground truth, 0 otherwise.

**Limitations:**
- **Sparse reward signal.** Early in GRPO training, the model rarely produces the exact correct answer, so nearly all completions receive 0. This makes learning very slow because GRPO has no gradient to follow when all group rewards are identical.
- **No partial credit.** A response with `<SOLUTION>14.0</SOLUTION>` when the ground truth is `14` receives 0, despite being semantically correct. Similarly, `<SOLUTION> 14 </SOLUTION>` with extra whitespace would also fail.
- **Ignores reasoning quality.** A correct answer with no working out is rewarded identically to one with detailed, correct reasoning. This fails to encourage the chain-of-thought behavior the system prompt asks for.
- **Brittle to formatting variations.** Numerical equivalences (e.g. `2/4` vs `0.5` vs `1/2`) are not handled.

---

## Submission Task 2: Design a Better Reward Function

The improved reward function, `reward_answer_quality`, addresses every limitation identified above by combining **multiple reward signals** into a single score (max = 5.0):

| Component | Points | What it addresses |
|---|---|---|
| **Format match** | +2.0 | Encourages correct structure, but not alone sufficient for full reward |
| **Non-empty answer** | +0.5 | Guards against empty `<SOLUTION></SOLUTION>` gaming |
| **Exact correctness** | +2.0 | Directly rewards the task objective |
| **Numeric proximity** (within 1% tolerance) | +1.5 | Partial credit for numerically equivalent answers like `14.0` vs `14` |
| **String similarity** (ratio >= 0.85) | +1.0 | Partial credit for minor formatting differences |
| **Reasoning bonus** (`<start_working_out>` block present) | +0.5 | Encourages chain-of-thought reasoning without making it the dominant signal |

**Key design principles:**

1. **Partial credit over binary feedback:** Instead of all-or-nothing, the reward has a smooth range from 0 to 5. This gives GRPO much richer advantage estimates within each generation group, speeding up learning.

2. **Correctness dominates format:** Format alone gives at most 2.5 out of 5.0 (format + non-empty). Achieving the remaining 2.0-2.5 requires actually getting the answer right. This prevents the model from gaming the reward through formatting alone.

3. **Robust answer matching:** The numeric proximity check (1% tolerance) handles cases like `14.0` vs `14`, or `3.14159` vs `3.1416`. The string similarity fallback catches other near-matches. This makes the reward less brittle than exact string matching.

4. **Reasoning is encouraged but not required:** The 0.5 bonus for having a `<start_working_out>` block encourages the model to show its work, but doesn't dominate the score. This avoids rewarding verbose but incorrect responses.

5. **Hard to game:** To get the maximum score, the model must produce correct format, a non-empty answer, the correct answer, AND reasoning. No single shortcut can exploit the full reward.

---

## Submission Task 3: Implement and Train with the New Reward

### Define all reward functions (baselines + improved)

In [26]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores


def reward_has_number(completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        has_digit = any(ch.isdigit() for ch in response)
        scores.append(1.0 if has_digit else 0.0)
    return scores


def check_answer_binary(prompts, completions, answer, **kwargs):
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [
        guess.group(1).strip()
        if (guess := match_format.search(r)) is not None else None
        for r in responses
    ]
    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        true_answer = str(true_answer).strip()
        is_correct = (guess is not None) and (guess == true_answer)
        scores.append(1.0 if is_correct else 0.0)
    return scores

### Improved multi-signal reward function

In [27]:
def reward_answer_quality(prompts, completions, answer, **kwargs):
    """
    Multi-signal reward combining format compliance, numeric proximity,
    and string similarity for richer GRPO gradient signal.

    Scoring breakdown (max = 5.0):
      +2.0  exact format match  (match_format found)
      +0.5  answer field non-empty
      +2.0  exact answer match
        OR
      +1.5  numeric match within 1% tolerance
        OR
      +1.0  high string similarity  (ratio >= 0.85)
      +0.5  bonus: reasoning present in <start_working_out> block
    """
    responses = [c[0]["content"] for c in completions]

    extracted = [
        m.group(1).strip() if (m := match_format.search(r)) is not None else None
        for r in responses
    ]

    scores = []
    for guess, true_answer, response in zip(extracted, answer, responses):
        score = 0.0
        true_str = str(true_answer).strip()

        if guess is not None:
            score += 2.0

            if guess:
                score += 0.5

                if guess == true_str:
                    score += 2.0
                else:
                    proximity_matched = False
                    try:
                        g_val = float(re.sub(r"[^\d.\-eE]", "", guess))
                        t_val = float(re.sub(r"[^\d.\-eE]", "", true_str))
                        if t_val != 0 and abs(g_val - t_val) / abs(t_val) <= 0.01:
                            score += 1.5
                            proximity_matched = True
                        elif t_val == 0 and abs(g_val) < 1e-6:
                            score += 1.5
                            proximity_matched = True
                    except ValueError:
                        pass

                    if not proximity_matched:
                        similarity = difflib.SequenceMatcher(
                            None, guess.lower(), true_str.lower()
                        ).ratio()
                        if similarity >= 0.85:
                            score += 1.0

        if re.search(r"<start_working_out>(.+?)<end_working_out>", response, re.DOTALL):
            score += 0.5

        scores.append(score)

    return scores

### Reward weighting and experiment configuration

In [28]:
def make_weighted_reward(func, weight: float):
    def wrapped(*args, **kwargs):
        scores = func(*args, **kwargs)
        return [weight * s for s in scores]
    wrapped.__name__ = f"{func.__name__}_w{weight}"
    return wrapped

In [29]:
EXPERIMENT_NAME = "improved_quality_reward"

experiment = {
    "label": "Improved: multi-signal quality reward",
    "description": (
        "GRPO fine-tuning with a multi-signal reward combining format compliance, "
        "numeric proximity, string similarity, and reasoning detection. "
        "Provides partial credit and richer gradient signal than the baselines."
    ),
    "reward_funcs": [
        make_weighted_reward(reward_answer_quality, 1.0),
    ],
    "checkpoint_dir": os.path.join(DRIVE_ROOT, "grpo_improved_quality_reward"),
}

reward_funcs = experiment["reward_funcs"]
CKPT_DIR = experiment["checkpoint_dir"]

print("=" * 80)
print(f"Experiment: {experiment['label']}")
print(f"Description: {experiment['description']}")
print(f"Checkpoint dir: {CKPT_DIR}")
print("Reward functions:", [f.__name__ for f in reward_funcs])
print("=" * 80)

Experiment: Improved: multi-signal quality reward
Description: GRPO fine-tuning with a multi-signal reward combining format compliance, numeric proximity, string similarity, and reasoning detection. Provides partial credit and richer gradient signal than the baselines.
Checkpoint dir: /content/drive/MyDrive/grpo_lab_checkpoints/grpo_improved_quality_reward
Reward functions: ['reward_answer_quality_w1.0']


In [30]:
os.makedirs(CKPT_DIR, exist_ok=True)

experiment_metadata = {
    "experiment_name": EXPERIMENT_NAME,
    "label": experiment["label"],
    "description": experiment["description"],
    "reward_functions": [f.__name__ for f in reward_funcs],
}
with open(os.path.join(CKPT_DIR, "experiment_metadata.json"), "w") as f:
    json.dump(experiment_metadata, f, indent=2)
print(f"Saved experiment metadata to {os.path.join(CKPT_DIR, 'experiment_metadata.json')}")

Saved experiment metadata to /content/drive/MyDrive/grpo_lab_checkpoints/grpo_improved_quality_reward/experiment_metadata.json


### GRPO Training

In [31]:
LAB_EXTRA_STEPS = 5
FRESH_RUN_MAX_STEPS = 15
RESUME_GRPO_IF_AVAILABLE = True
LAB_SAVE_STEPS = 5

print("=" * 80)
print(f"LAB_EXTRA_STEPS = {LAB_EXTRA_STEPS}")
print(f"FRESH_RUN_MAX_STEPS = {FRESH_RUN_MAX_STEPS}")
print(f"RESUME_GRPO_IF_AVAILABLE = {RESUME_GRPO_IF_AVAILABLE}")
print(f"LAB_SAVE_STEPS = {LAB_SAVE_STEPS}")
print("=" * 80)

LAB_EXTRA_STEPS = 5
FRESH_RUN_MAX_STEPS = 15
RESUME_GRPO_IF_AVAILABLE = True
LAB_SAVE_STEPS = 5


In [32]:
LATEST_CKPT = get_latest_checkpoint(CKPT_DIR) if RESUME_GRPO_IF_AVAILABLE else None
START_STEP = get_checkpoint_step(LATEST_CKPT)

if LATEST_CKPT is not None:
    TARGET_MAX_STEPS = START_STEP + LAB_EXTRA_STEPS
    EFFECTIVE_SAVE_STEPS = min(LAB_SAVE_STEPS, LAB_EXTRA_STEPS)
    RUN_MODE = "resume"
else:
    TARGET_MAX_STEPS = FRESH_RUN_MAX_STEPS
    EFFECTIVE_SAVE_STEPS = LAB_SAVE_STEPS
    RUN_MODE = "fresh_start"

print("=" * 80)
print(f"Run mode: {RUN_MODE}")
print(f"Latest checkpoint: {LATEST_CKPT}")
print(f"Start step: {START_STEP}")
print(f"Target max steps: {TARGET_MAX_STEPS}")
print(f"Save every: {EFFECTIVE_SAVE_STEPS} steps")
print("=" * 80)

Run mode: fresh_start
Latest checkpoint: None
Start step: 0
Target max steps: 15
Save every: 5 steps


In [33]:
max_prompt_length = maximum_length + 1
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1,
    num_generations = 2,
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = TARGET_MAX_STEPS,
    save_steps = EFFECTIVE_SAVE_STEPS,
    save_total_limit = 3,
    report_to = "none",
    output_dir = CKPT_DIR,
)

print("=" * 80)
print(f"GRPO output_dir: {CKPT_DIR}")
print(f"GRPO max_steps: {TARGET_MAX_STEPS}")
print(f"GRPO save_steps: {EFFECTIVE_SAVE_STEPS}")
print("=" * 80)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 2
GRPO output_dir: /content/drive/MyDrive/grpo_lab_checkpoints/grpo_improved_quality_reward
GRPO max_steps: 15
GRPO save_steps: 5


In [34]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = reward_funcs,
    args = training_args,
    train_dataset = dataset,
)

print("=" * 80)
print(f"Starting GRPO run: {experiment['label']}")
print(f"Description: {experiment['description']}")
print(f"Run mode: {RUN_MODE}")
if LATEST_CKPT is not None:
    print(f"Resuming from checkpoint: {LATEST_CKPT}")
    print(f"Continuing from step {START_STEP} to step {TARGET_MAX_STEPS}")
else:
    print("No GRPO checkpoint found. Starting fresh from the SFT-only baseline.")
    print(f"Training to step {TARGET_MAX_STEPS}")
print("=" * 80)

trainer.train(
    resume_from_checkpoint = LATEST_CKPT if LATEST_CKPT is not None else None
)

Starting GRPO run: Improved: multi-signal quality reward
Description: GRPO fine-tuning with a multi-signal reward combining format compliance, numeric proximity, string similarity, and reasoning detection. Provides partial credit and richer gradient signal than the baselines.
Run mode: fresh_start
No GRPO checkpoint found. Starting fresh from the SFT-only baseline.
Training to step 15


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,709 | Num Epochs = 1 | Total steps = 15
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 32768}. If this is not desired, please set these values explicitly.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_answer_quality_w1.0 / mean,rewards / reward_answer_quality_w1.0 / std
1,0.000100,0.000000,0.000000,1846.000000,1846.000000,1846.000000,1.000000,0.000000,0.000000,0.000000,0.144118,0.000000,0.000000
2,0.101700,1.250000,1.767767,1706.500000,1567.000000,1846.000000,0.500000,1567.000000,1567.000000,1567.000000,101.745575,1.250000,1.767767
3,0.000100,0.000000,0.000000,1846.000000,1846.000000,1846.000000,1.000000,0.000000,0.000000,0.000000,0.132304,0.000000,0.000000
4,0.000100,0.000000,0.000000,1846.000000,1846.000000,1846.000000,1.000000,0.000000,0.000000,0.000000,0.101778,0.000000,0.000000
5,0.015600,4.500000,0.000000,1340.500000,1021.000000,1660.000000,0.000000,1340.500000,1021.000000,1660.000000,15.622718,4.500000,0.000000
6,0.000100,0.000000,0.000000,1846.000000,1846.000000,1846.000000,1.000000,0.000000,0.000000,0.000000,0.103285,0.000000,0.000000
7,0.000100,0.000000,0.000000,1846.000000,1846.000000,1846.000000,1.000000,0.000000,0.000000,0.000000,0.107472,0.000000,0.000000
8,0.000100,0.000000,0.000000,1846.000000,1846.000000,1846.000000,1.000000,0.000000,0.000000,0.000000,0.106195,0.000000,0.000000
9,0.000100,0.000000,0.000000,1846.000000,1846.000000,1846.000000,1.000000,0.000000,0.000000,0.000000,0.115538,0.000000,0.000000
10,0.000100,0.000000,0.000000,1846.000000,1846.000000,1846.000000,1.000000,0.000000,0.000000,0.000000,0.104960,0.000000,0.000000


TrainOutput(global_step=15, training_loss=0.008011781044964058, metrics={'train_runtime': 2993.8256, 'train_samples_per_second': 0.01, 'train_steps_per_second': 0.005, 'total_flos': 0.0, 'train_loss': 0.008011781044964058})

### Save Final LoRA Adapter

In [35]:
FINAL_LORA_DIR = os.path.join(CKPT_DIR, "final_lora")
os.makedirs(FINAL_LORA_DIR, exist_ok=True)

model.save_pretrained(FINAL_LORA_DIR)
tokenizer.save_pretrained(FINAL_LORA_DIR)

del model
del tokenizer
del trainer
torch.cuda.empty_cache()

print("=" * 80)
print(f"Saved final LoRA adapter to: {FINAL_LORA_DIR}")
print(f"Experiment: {experiment['label']}")
print("=" * 80)

Saved final LoRA adapter to: /content/drive/MyDrive/grpo_lab_checkpoints/grpo_improved_quality_reward/final_lora
Experiment: Improved: multi-signal quality reward


In [36]:
from safetensors import safe_open

with safe_open(os.path.join(FINAL_LORA_DIR, "adapter_model.safetensors"), framework="pt") as f:
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert (n_zeros.item() != tensor.numel())

print(f"Verified adapter weights in {FINAL_LORA_DIR}")

Verified adapter weights in /content/drive/MyDrive/grpo_lab_checkpoints/grpo_improved_quality_reward/final_lora


---

## Submission Task 4: Evaluate and Compare Against Baselines

In [37]:
CHECKPOINT_OPTIONS = {
    "baseline": {
        "path": os.path.join(DRIVE_ROOT, "sft_only_baseline"),
        "label": "SFT-only baseline",
        "description": "Model after supervised fine-tuning only; no RL fine-tuning applied.",
    },
    "bad_format_only": {
        "path": os.path.join(DRIVE_ROOT, "grpo_bad_format_only", "final_lora"),
        "label": "Bad reward baseline 1: format-only",
        "description": "GRPO model trained with a reward that checks only whether the response follows the expected format.",
    },
    "bad_has_number": {
        "path": os.path.join(DRIVE_ROOT, "grpo_bad_has_number", "final_lora"),
        "label": "Bad reward baseline 2: has-a-number",
        "description": "GRPO model trained with a reward that checks only whether the response contains a number.",
    },
    "better_binary_reward": {
        "path": os.path.join(DRIVE_ROOT, "grpo_better_binary_reward", "final_lora"),
        "label": "Slightly better baseline: binary correctness",
        "description": "GRPO model trained with a binary correctness reward.",
    },
    "improved_quality_reward": {
        "path": os.path.join(DRIVE_ROOT, "grpo_improved_quality_reward", "final_lora"),
        "label": "Improved: multi-signal quality reward",
        "description": "GRPO model trained with multi-signal reward (format + proximity + similarity + reasoning).",
    },
}

test_questions = [
    "What is the sqrt of 101?",
    "If a train travels 120 km in 1.5 hours, what is its speed in km/h?",
    "Solve for x: 3x + 7 = 22",
]

In [41]:
import tempfile

for ckpt_name, ckpt_info in CHECKPOINT_OPTIONS.items():
    if not os.path.exists(ckpt_info["path"]):
        print(f"[SKIP] {ckpt_info['label']} — checkpoint not found at {ckpt_info['path']}")
        continue

    print("\n" + "=" * 80)
    print(f"Loading: {ckpt_info['label']}")
    print(f"Description: {ckpt_info['description']}")
    print(f"Path: {ckpt_info['path']}")
    print("=" * 80)

    # The `offload_dir` parameter is not supported by FastLanguageModel for Qwen3
    # and is causing a TypeError. Unsloth handles memory internally.
    # Reducing max_seq_length to conserve VRAM on T4, as 4bit loading alone might not be enough.
    infer_model, infer_tokenizer = FastLanguageModel.from_pretrained(
        model_name = ckpt_info["path"],
        max_seq_length = 1024, # Reduced from global max_seq_length (2048) to conserve VRAM
        load_in_4bit = True,
        fast_inference = False,
        max_lora_rank = lora_rank,
        gpu_memory_utilization = 0.9,
    )
    FastLanguageModel.for_inference(infer_model)

    for question in test_questions:
        print(f"\n--- Question: {question} ---")
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question},
        ]
        text = infer_tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False,
        )
        # Determine appropriate max_new_tokens based on actual prompt length and new max_seq_length
        prompt_tokens = infer_tokenizer(text, return_tensors="pt").to("cuda")['input_ids'].shape[1]
        effective_max_new_tokens = max(1, 1024 - prompt_tokens) # Ensure at least 1 token is generated, don't exceed model's max_seq_length

        _ = infer_model.generate(
            **infer_tokenizer(text, return_tensors="pt").to("cuda"),
            temperature = 1.0,
            top_k = 50,
            max_new_tokens = effective_max_new_tokens,
            streamer = TextStreamer(infer_tokenizer, skip_prompt=False),
        )

    del infer_model
    del infer_tokenizer
    torch.cuda.empty_cache()
    print("\n")


Loading: SFT-only baseline
Description: Model after supervised fine-tuning only; no RL fine-tuning applied.
Path: /content/drive/MyDrive/grpo_lab_checkpoints/sft_only_baseline
==((====))==  Unsloth 2026.4.1: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.32G [00:00<?, ?B/s]

RuntimeError: CUDA error: an illegal memory access was encountered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


### Evaluation Discussion

After running inference across all checkpoints, compare the outputs using these criteria:

| Criterion | Format-only | Has-number | Binary | **Improved (ours)** |
|---|---|---|---|---|
| Follows required format? | Yes (rewarded directly) | Sometimes | Sometimes | Yes (2.0 reward component) |
| Correct final answer? | Rarely — not rewarded | Rarely — not rewarded | Sometimes — but sparse signal | More often — partial credit helps learning |
| Shows reasoning? | Incidental | Incidental | Incidental | Encouraged (+0.5 bonus) |
| Handles numeric equivalence? | No | No | No | Yes (1% tolerance) |
| Resistant to gaming? | No (just emit tags) | No (any digit works) | Moderate | Strong (needs format + correctness + reasoning) |

**Expected outcomes:**

- **Format-only** will consistently produce the `<SOLUTION>` tags but with low answer accuracy, since formatting is the only rewarded behavior.
- **Has-a-number** will show minimal behavioral change from the SFT baseline, because nearly all outputs already contain digits, so the reward provides no useful learning signal.
- **Binary correctness** will show some improvement in accuracy over the weak baselines, but learning will be slow due to the sparse (0 or 1) reward.
- **Improved quality reward** should show the best combination of format compliance AND answer accuracy, because:
  - Partial credit creates useful gradients even when the answer is close but not exact
  - The multi-signal structure ensures the model is simultaneously rewarded for reasoning, formatting, and correctness
  - Numeric proximity prevents the model from being penalized for trivial formatting differences like `14.0` vs `14`